# Checkpoint 5G — AMPds2 Remaining Seven Windows: Five-Seed Kernel SHAP

Run the **single code cell below in the existing verified Colab runtime** after Checkpoint 5F. Do not run earlier notebooks again. This is a controlled five-seed reconstruction, **not proof of exact historical Kernel SHAP attribution identity**.

The code checks the locked protocol, the original model and 13-window evidence hashes, all 100 background scores for each of the five seeds, and SHAP additivity. It runs 35 explanations (`AMPDS2_02`–`AMPDS2_08`), preserving 805 original-feature values, 70 pairwise comparisons and seven window summaries only after all checks pass. Historical artifacts are never overwritten.


In [ ]:
# CHECKPOINT 5G — AMPds2 remaining SEVEN windows: five-seed Kernel SHAP
# Frozen Checkpoint 4B protocol. Only background-sampling seed varies.
# No model fitting, LIME execution, threshold changes, or historical overwrites.

from pathlib import Path
from itertools import combinations
import hashlib
import importlib.metadata as im
import json
import joblib
import numpy as np
import pandas as pd
import shap

# A. Immutable protocol / environment gates
required = ['ROOT','D5','FEATURES','COLUMNS','STABILITY_PROTOCOL','protocol_hash']
missing = [key for key in required if key not in globals()]
assert not missing, f'STOP: Missing verified Colab variables: {missing}'

for pkg, version in {'scikit-learn':'1.6.1','shap':'0.54.1',
                     'numpy':'2.4.6','pandas':'2.2.3'}.items():
    assert im.version(pkg) == version, f'STOP: {pkg} version changed'

ROOT5G, D55G = Path(ROOT), Path(D5)
seeds5g = [42,123,456,789,2026]
case_ids5g = [f'AMPDS2_{i:02d}' for i in range(2,9)]
features5g = list(FEATURES)
columns5g = list(COLUMNS)
assert len(features5g) == 23 and columns5g == [f'{name}_z' for name in features5g]

p5g = STABILITY_PROTOCOL
canonical5g = json.dumps(p5g,sort_keys=True,separators=(',',':'))
expected_protocol_hash5g = '849e9d7ef15cc3b135ac74854f491ae3c2113f39c3ef527ca789cde8e829334c'
assert hashlib.sha256(canonical5g.encode('utf-8')).hexdigest() == protocol_hash
assert protocol_hash == expected_protocol_hash5g
assert p5g['seeds'] == seeds5g and p5g['feature_order'] == features5g
assert p5g['datasets']['AMPds2'] == 8
scfg5g = p5g['kernel_shap']
assert scfg5g['vary'] == 'background_sampling_seed'
assert scfg5g['training_order']['AMPds2'] == 'scoring_csv'
assert scfg5g['background_size'] == 100
assert scfg5g['nsamples'] == 300
assert scfg5g['coalition_rng_seed'] == 42
assert scfg5g['l1_reg'] == 'num_features(10)'
assert scfg5g['additivity_tolerance'] == 1e-6

protocol_path5g = (ROOT5G/'SUPERVISOR_FINAL_EXPERIMENTS_2026'/'01_protocol'/
                   'CHECKPOINT4B_STABILITY_13WINDOW_849e9d7e.json')
assert protocol_path5g.is_file()
assert hashlib.sha256(protocol_path5g.read_bytes()).hexdigest() == expected_protocol_hash5g

def file_sha256_5g(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def raw_sha256_5g(data):
    return hashlib.sha256(data).hexdigest()

manifest5g = json.loads((D55G/'manifest.json').read_text(encoding='utf-8'))
entry5g = manifest5g['model_and_background']['AMPds2']
histbg5g = entry5g['shap_background_verification']
assert histbg5g['status'] == 'NUMERICALLY_CONFIRMED'
assert histbg5g['training_order'] == 'scoring_csv'
assert histbg5g['background_size'] == 100 and histbg5g['sampling_seed'] == 42
model_path5g = Path(entry5g['model_path'])
score_path5g = Path(entry5g['scoring_file'])
selected_path5g = D55G/'selected_scaled_inputs_13_windows.csv'
assert model_path5g.is_file() and score_path5g.is_file() and selected_path5g.is_file()
assert file_sha256_5g(model_path5g) == entry5g['model_sha256']
assert file_sha256_5g(selected_path5g) == manifest5g['artifact_sha256'][selected_path5g.name]

# User-owned, SHA-256-verified historical reconstruction; no model fitting.
model5g = joblib.load(model_path5g)
assert model5g.n_features_in_ == 23
assert len(model5g.estimators_) == 300 and model5g.max_samples_ == 256
assert abs(float(model5g.offset_)+0.5) <= 1e-12

def anomaly_score5g(a):
    mat=np.asarray(a,dtype=np.float32)
    assert mat.ndim==2 and mat.shape[1]==23 and np.isfinite(mat).all()
    return -model5g.decision_function(mat)

selected5g = pd.read_csv(selected_path5g,float_precision='round_trip')
assert len(selected5g)==13 and selected5g.window_id.is_unique
assert selected5g.dataset.value_counts().to_dict() == {'UK-DALE':5,'AMPds2':8}
cases5g = selected5g.loc[selected5g.window_id.isin(case_ids5g)].copy()
assert len(cases5g)==7 and set(cases5g.window_id)==set(case_ids5g)
assert cases5g.dataset.eq('AMPds2').all()
for case_id in case_ids5g:
    case=cases5g.loc[cases5g.window_id.eq(case_id)]
    assert len(case)==1
    X=case[columns5g].to_numpy(dtype=np.float32)
    assert X.shape==(1,23) and np.isfinite(X).all()
    assert abs(float(anomaly_score5g(X)[0])-float(case.iloc[0]['anomaly_score'])) <= 1e-9

# Verify the completed AMPDS2_01 SHAP pilot has not been changed.
parent5g = ROOT5G/'SUPERVISOR_FINAL_EXPERIMENTS_2026'/'06_explanation_evaluation'
assert parent5g.is_dir()
prior5g = (parent5g/'CHECKPOINT5C_AMPDS201_SHAP_BACKGROUND_5SEEDS_849e9d7e'/
           'ampds201_shap_runs.csv')
assert prior5g.is_file()
prior_runs5g = pd.read_csv(prior5g)
assert len(prior_runs5g)==5
assert set(prior_runs5g.seed)==set(seeds5g)
assert prior_runs5g.window_id.eq('AMPDS2_01').all()
assert prior_runs5g.additivity_residual.max() <= 1e-6
print('PASS: Frozen protocol, model, original 13-window file and seven AMPds2 target scores')

# B. Independently reconstruct all five background samples in one CSV scan.
# Training order is EXACTLY as it appeared in the corrected scoring CSV.
N_TRAIN5G=441483
positions5g={s:np.random.default_rng(s).choice(N_TRAIN5G,size=100,replace=False)
             for s in seeds5g}
needed5g=sorted(set(int(q) for a in positions5g.values() for q in a))
sampled5g={}
seen5g=0
for chunk in pd.read_csv(score_path5g,
                         usecols=['split','anomaly_score']+columns5g,
                         chunksize=25000,float_precision='round_trip',low_memory=False):
    train_chunk = chunk.loc[chunk['split'].astype(str).eq('train')]
    n=len(train_chunk)
    if n==0: continue
    hits=[q for q in needed5g if seen5g <= q < seen5g+n]
    if hits:
        subset=train_chunk.iloc[[q-seen5g for q in hits]]
        vectors=subset[columns5g].to_numpy(dtype=np.float32)
        old_scores=subset['anomaly_score'].to_numpy(dtype=np.float64)
        for pos, vector, stored in zip(hits,vectors,old_scores):
            sampled5g[pos]=(vector,float(stored))
    seen5g += n
assert seen5g==N_TRAIN5G and len(sampled5g)==len(needed5g)

backgrounds5g={}
for s in seeds5g:
    positions=positions5g[s]
    bg=np.stack([sampled5g[int(q)][0] for q in positions]).astype(np.float32)
    archived_bg_scores=np.asarray([sampled5g[int(q)][1] for q in positions])
    assert bg.shape==(100,23) and np.isfinite(bg).all()
    assert np.max(np.abs(anomaly_score5g(bg)-archived_bg_scores)) <= 1e-9
    backgrounds5g[s]=bg

assert raw_sha256_5g(backgrounds5g[42].tobytes(order='C'))==histbg5g['background_sha256']
assert raw_sha256_5g(positions5g[42].astype('<i8').tobytes())==histbg5g['sample_positions_sha256']
assert abs(float(anomaly_score5g(backgrounds5g[42]).mean())-float(histbg5g['expected_value'])) <= 1e-9
print(f'PASS: {seen5g:,} original training rows, {len(needed5g)} unique sampled positions')
print('PASS: Original seed-42 background hash, index hash, and all 500 sampled scores')

# C. Run exactly seven windows x five fixed background samples.
# Fixed seed42 coalition sampling is reset BEFORE EVERY new SHAP explanation.
runs5g, vectors5g, pairs5g, summaries5g = [],[],[],[]
old_numpy_state5g=np.random.get_state()
try:
    for case_id in case_ids5g:
        case=cases5g.loc[cases5g.window_id.eq(case_id)]
        X=case[columns5g].to_numpy(dtype=np.float32)
        target=float(anomaly_score5g(X)[0])
        top_sets={}
        residuals=[]
        for s in seeds5g:
            bg=backgrounds5g[s]
            np.random.seed(42)
            explainer=shap.KernelExplainer(anomaly_score5g,bg)
            phi=np.asarray(explainer.shap_values(
                X,nsamples=300,l1_reg='num_features(10)'),dtype=np.float64).reshape(-1)
            baseline=float(np.asarray(explainer.expected_value).reshape(-1)[0])
            assert phi.shape==(23,) and np.isfinite(phi).all()
            assert abs(baseline-float(anomaly_score5g(bg).mean()))<=1e-9
            residual=abs(baseline+float(phi.sum())-target)
            assert residual <= 1e-6, f'STOP: Additivity failure {case_id}, seed {s}'
            order=np.lexsort((np.arange(23),-np.abs(phi)))
            top=order[:5].tolist()
            top_sets[s]=set(top)
            residuals.append(residual)
            runs5g.append(dict(dataset='AMPds2',window_id=case_id,seed=s,
                score=target,expected_value=baseline,additivity_residual=residual,
                background_sha256=raw_sha256_5g(bg.tobytes(order='C')),
                positions_sha256=raw_sha256_5g(positions5g[s].astype('<i8').tobytes()),
                nonzero_contributions=int(np.count_nonzero(phi)),
                top5_features='|'.join(features5g[i] for i in top)))
            for i,feature in enumerate(features5g):
                vectors5g.append(dict(dataset='AMPds2',window_id=case_id,seed=s,
                    feature_index=i,feature=feature,scaled_feature_value=float(X[0,i]),
                    shap_value=float(phi[i]),abs_shap_value=float(abs(phi[i])),
                    top5=i in top_sets[s]))
            print(f'PASS {case_id} seed={s}: residual={residual:.3e}; top5={[features5g[i] for i in top]}')
        window_j=[]
        for a,b in combinations(seeds5g,2):
            A,B=top_sets[a],top_sets[b]
            j=len(A&B)/len(A|B)
            window_j.append(j)
            pairs5g.append(dict(dataset='AMPds2',window_id=case_id,seed_a=a,seed_b=b,
                intersection_count=len(A&B),union_count=len(A|B),top5_jaccard=j))
        assert len(window_j)==10
        summaries5g.append(dict(dataset='AMPds2',window_id=case_id,
            protocol_sha256=protocol_hash,method='Kernel SHAP',
            vary='background_sampling_seed',coalition_rng_seed=42,
            historical_attribution_identity='NOT_ESTABLISHED',n_runs=5,n_seed_pairs=10,
            mean_pairwise_top5_jaccard=float(np.mean(window_j)),
            min_pairwise_top5_jaccard=float(np.min(window_j)),
            max_pairwise_top5_jaccard=float(np.max(window_j)),
            max_additivity_residual=float(max(residuals))))
finally:
    np.random.set_state(old_numpy_state5g)

assert len(runs5g)==35 and len(vectors5g)==805 and len(pairs5g)==70
assert len(summaries5g)==7
assert {r['window_id'] for r in runs5g}==set(case_ids5g)
assert all(len([z for z in runs5g if z['window_id']==wid])==5 for wid in case_ids5g)
assert max(z['additivity_residual'] for z in runs5g)<=1e-6

# D. Preserve only NEW evidence, without overwriting different existing bytes.
output5g=parent5g/'CHECKPOINT5G_AMPDS2_REMAINING7_SHAP_5SEEDS_849e9d7e'
def csv_bytes5g(rows):
    return pd.DataFrame(rows).to_csv(index=False,float_format='%.17g').encode('utf-8')

payloads5g={
    'ampds2_remaining7_shap_runs.csv':csv_bytes5g(runs5g),
    'ampds2_remaining7_shap_23_features_by_seed.csv':csv_bytes5g(vectors5g),
    'ampds2_remaining7_shap_pairwise_jaccard.csv':csv_bytes5g(pairs5g),
    'ampds2_remaining7_shap_window_summary.csv':csv_bytes5g(summaries5g),
}
manifest_out5g = {
    'checkpoint':'5G', 'dataset':'AMPds2','method':'Kernel SHAP',
    'window_ids':case_ids5g,'seeds':seeds5g,
    'treatment':'background_sampling_seed_only',
    'coalition_rng_seed':42, 'protocol_sha256':protocol_hash,
    'model_sha256':entry5g['model_sha256'],
    'selected_input_sha256':manifest5g['artifact_sha256'][selected_path5g.name],
    'historical_attribution_identity':'NOT_ESTABLISHED',
    'n_runs':len(runs5g),'n_full_feature_records':len(vectors5g),
    'n_pairwise_comparisons':len(pairs5g),
    'historical_files_unchanged':True,
    'output_sha256':{key:raw_sha256_5g(value) for key,value in payloads5g.items()}
}
payloads5g['ampds2_remaining7_shap_batch_manifest.json']=(
    json.dumps(manifest_out5g,sort_keys=True,indent=2,allow_nan=False)+'\n').encode('utf-8')
for filename,data in payloads5g.items():
    path=output5g/filename
    if path.exists():
        assert path.read_bytes()==data, f'STOP: Existing evidence differs: {path}'
output5g.mkdir(exist_ok=True)
for filename,data in payloads5g.items():
    path=output5g/filename
    if not path.exists():
        with path.open('xb') as stream:stream.write(data)
    assert path.read_bytes()==data

print('\n=== CHECKPOINT 5G SUMMARY ===')
print(pd.DataFrame(summaries5g)[['window_id','mean_pairwise_top5_jaccard',
    'min_pairwise_top5_jaccard','max_additivity_residual']].to_string(index=False))
print('Total verified explanations:',len(runs5g))
print('Total original-feature attribution records:',len(vectors5g))
print('Total Jaccard comparisons:',len(pairs5g))
print('Evidence directory:',output5g)
print('PASS: AMPDS2_REMAINING7_SHAP_FIVE_SEED_EXPERIMENT')
print('No detector retraining, no LIME, no threshold changes, no historical overwrites.')
